# Neighborhood Enrichment Analysis with Squidpy

Companion notebook for: [spatiabio.com — post 7](https://www.spatiabio.com)

Which Leiden clusters co-locate spatially on 10x Genomics Visium mouse brain data?
We use `sq.gr.nhood_enrichment` to compute permutation-based z-scores for all cluster pairs.

In [ ]:
import squidpy as sq
import scanpy as sc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Load data and preprocess

In [ ]:
adata = sq.datasets.visium_hne_adata()

# NOTE: visium_hne_adata() ships .X ALREADY log-normalized (max ~8.4, non-integer);
# the raw integer counts live in adata.raw.X. Calling normalize_total + log1p here
# would log it a second time -- silently, with no error, changing every result below.
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='cell_ranger')
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.leiden(adata, key_added='cluster')  # pip install leidenalg if needed

print(f'Clusters: {adata.obs["cluster"].nunique()}')
print(adata.obs['cluster'].value_counts())

## 2. Build spatial neighbor graph and run neighborhood enrichment

In [ ]:
sq.gr.spatial_neighbors(adata)
sq.gr.nhood_enrichment(adata, cluster_key='cluster')
print('Done. Results in adata.uns["cluster_nhood_enrichment"]')

## 3. Visualize clusters on tissue

In [ ]:
sq.pl.spatial_scatter(adata, color='cluster', title='Leiden clusters on tissue')

## 4. Neighborhood enrichment heatmap

In [ ]:
sq.pl.nhood_enrichment(
    adata,
    cluster_key='cluster',
    title='Neighborhood enrichment (Leiden clusters)'
)

## 5. Extract top co-localizing pairs

In [ ]:
zscore = adata.uns['cluster_nhood_enrichment']['zscore']
n = zscore.shape[0]
rows = []
for i in range(n):
    for j in range(i+1, n):
        rows.append(dict(cluster_i=i, cluster_j=j, zscore=zscore[i, j]))

df = pd.DataFrame(rows).sort_values('zscore', ascending=False)
print('Top co-localizing pairs (off-diagonal):')
print(df[df.zscore > 0].to_string(index=False))